In [ ]:
import sys
sys.path.insert(0, "../..")

import glob
import json
import os

import joblib
from tqdm import tqdm

In [ ]:
DATA_ROOT = "../../data"

REDDIT_INPUT  = os.path.join(DATA_ROOT, "reddit/03_prepared_data")
TG_INPUT      = os.path.join(DATA_ROOT, "telegram/03_prepared_data")

GEN_INPUT_DIR = os.path.join(DATA_ROOT, "generation_input")
GEN_RESULTS_DIR = os.path.join(DATA_ROOT, "generation_output")

MODELS = {
    "qwen":        "Qwen/Qwen3-235B-A22B-Instruct-2507-tput",
    "gemma_small": "google/gemma-3n-E4B-it",
}

MODEL_EXTRA_PARAMS = {
    "gemma_large": {"reasoning": {"enabled": False}},
}

PROMPT_TYPES = {
    "full":    "prompt",
    "reduced": "reduced_prompt",
}

MAX_BATCH_SIZE = 5000
MAX_TOKENS     = 2000
TEMPERATURE    = 0.5
SEED           = 42

In [ ]:
reddit_files = glob.glob(os.path.join(REDDIT_INPUT, "raw*.joblib"))
tg_files     = glob.glob(os.path.join(TG_INPUT,     "raw*.joblib"))

all_data_files = [
    f for f in reddit_files + tg_files
    if "mapping" not in f
]

print(f"Found {len(all_data_files)} input files.")
all_data_files

In [ ]:
def already_processed_channels(results_dir: str) -> set:
    """Return channel names that already have result files in results_dir."""
    existing = glob.glob(os.path.join(results_dir, "*"))
    return {os.path.basename(f).split("_")[-2] for f in existing}

In [ ]:
for model_key, model_name in MODELS.items():
    print(f"\n{'='*64}")
    print(f"Model: {model_key}  ({model_name})")

    for prompt_type, prompt_field in PROMPT_TYPES.items():
        output_dir  = os.path.join(GEN_INPUT_DIR,  model_key, prompt_type)
        results_dir = os.path.join(GEN_RESULTS_DIR, model_key, prompt_type)
        os.makedirs(output_dir, exist_ok=True)

        done_channels = already_processed_channels(results_dir)

        remaining = [
            f for f in all_data_files
            if f.split("_")[-1].replace(".joblib", "") not in done_channels
        ]

        print(f"\n  [{prompt_type}] output → {output_dir}")
        print(f"  Already done: {len(done_channels)} | Remaining: {len(remaining)}/{len(all_data_files)}")

        extra_params = MODEL_EXTRA_PARAMS.get(model_key, {})

        created = []
        for data_file in tqdm(remaining, desc=f"{model_key}/{prompt_type}"):
            base_name = (
                os.path.basename(data_file)
                .replace(".joblib", "")
                .replace("raw_generation", model_key)
            )

            data = joblib.load(data_file)
            print(f"    {base_name}: {len(data)} records")

            for batch_start in range(0, len(data), MAX_BATCH_SIZE):
                out_path = os.path.join(output_dir, f"{base_name}_{batch_start}.jsonl")
                with open(out_path, "w") as fh:
                    for record in data[batch_start : batch_start + MAX_BATCH_SIZE]:
                        prompt_text = record[prompt_field]
                        request = {
                            "custom_id": f"{model_key}_{prompt_type}_{record['user_id']}_{record['thread_id']}",
                            "body": {
                                "model": model_name,
                                "messages": [
                                    {"role": "user", "content": prompt_text}
                                ],
                                "max_tokens": MAX_TOKENS,
                                "temperature": TEMPERATURE,
                                "seed": SEED,
                                **extra_params,
                            },
                        }
                        fh.write(json.dumps(request) + "\n")
                created.append(out_path)

        print(f"{len(created)} JSONL files written.")